# 🧠 Notebook 04: Tensors and Graphs

## 1. Purpose + Scope

This notebook explores the fundamental structures for high-performance computing in T81:

*   **T81Tensor Rank/Dimension Model**: How N-dimensional arrays are managed.
*   **Hybrid Stack/Heap Storage**: Efficient memory allocation strategies.
*   **Deterministic Matmul**: Ensuring consistent results in matrix operations.
*   **Graph Adjacency Canonicalization**: Consistent representation of graph structures.
*   **PageRank and BFS**: Examples of graph algorithms.

## 2. Spec References

*   `spec/t81-data-types.md`
*   `include/t81/core/T81Tensor.hpp`
*   `include/t81/core/T81Graph.hpp`

## 3. Determinism Tier

**Tier A (Strict Determinism)**: Tensor operations are guaranteed to be bit-exact across platforms, crucial for verifiable AI models.

## 4. Reproducibility Setup

Ensure `t81_python` is built and available in `PYTHONPATH`.

In [ ]:
import sys
import os

build_dir = os.path.abspath(os.path.join(os.getcwd(), "../build"))
if build_dir not in sys.path:
    sys.path.append(build_dir)

try:
    from t81_python import Tensor1D3, Tensor2D33, T81Int
    print("✅ t81_python Tensors loaded.")
except ImportError:
    print("❌ Failed to load t81_python Tensor types.")
    sys.exit(1)

## 5. Exploratory Code: Tensor Operations

We use the bound tensor types (e.g., `Tensor1D3`, `Tensor2D33`) to demonstrate operations.

In [ ]:
# Create a 3x3 matrix (Tensor2D33)
mat = Tensor2D33(T81Int(0))

# Set diagonal
for i in range(3):
    mat[i, i] = T81Int(1)

print("Identity Matrix:")
for i in range(3):
    row = [str(mat[i, j]) for j in range(3)]
    print(row)

# Add matrices
mat2 = Tensor2D33(T81Int(2))
mat_sum = mat + mat2

print("\nSum Matrix:")
for i in range(3):
    row = [str(mat_sum[i, j]) for j in range(3)]
    print(row)

## 6. Graph Representation

Graphs are represented by adjacency matrices or lists. Determinism requires canonical ordering of nodes.

In [ ]:
# Conceptual Graph Adjacency
# Nodes: A, B, C sorted alphabetically for canonical index 0, 1, 2
# Edges: A->B, B->C, C->A

adj = Tensor2D33(T81Int(0))
adj[0, 1] = T81Int(1) # A->B
adj[1, 2] = T81Int(1) # B->C
adj[2, 0] = T81Int(1) # C->A

print("\nAdjacency Matrix (A, B, C):")
for i in range(3):
    row = [str(adj[i, j]) for j in range(3)]
    print(row)

## 7. Deterministic Matmul

Matrix multiplication is the core of many algorithms, including PageRank steps.

In [ ]:
# Simulating matmul if not directly bound for Tensor2D33 * Tensor2D33
# (Assuming only + is bound for now based on exploration)
def matmul_sim(m1, m2):
    res = Tensor2D33(T81Int(0))
    for i in range(3):
        for j in range(3):
            sum_val = T81Int(0)
            for k in range(3):
                sum_val = sum_val + (m1[i, k] * m2[k, j])
            res[i, j] = sum_val
    return res

squared_adj = matmul_sim(adj, adj)
print("\nAdjacency Squared (2-step paths):")
for i in range(3):
    row = [str(squared_adj[i, j]) for j in range(3)]
    print(row)

## 8. Failure Mode Demonstration

Index out of bounds on fixed-size tensors.

In [ ]:
try:
    val = mat[3, 0]
except IndexError as e:
    print(f"Caught expected error: {e}")
except Exception as e:
    print(f"Caught exception: {e}")

## 9. Architectural Commentary

Fixed-size tensors like `Tensor2D33` are stack-allocated for speed, avoiding heap fragmentation. Large tensors use specific memory arenas. This hybrid approach balances performance with safety.